# GRPO on Qwen2.5-0.5B (v2): teaching a small model to reason with RL

Train Qwen2.5-0.5B-Instruct on GSM8K using GRPO (reward = right/wrong answer + format), then compare against the base model on the **full GSM8K test set (1,319 questions)**.

v2 vs v1: 4x longer training (1000 steps), higher LoRA learning rate and rank, fairer eval budget (512 tokens), accuracy tracked during training.

**Kaggle settings before running:** Accelerator = GPU T4 x2 (we use one), Internet = ON.
Run with **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab (~8h total, under Kaggle's 12h limit).

In [ ]:
# Use a single GPU (Trainer + 2 GPUs without accelerate launch causes issues)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U trl peft accelerate datasets
!pip uninstall -y -q torchao   # Kaggle's old torchao breaks new peft; we don't need it
# If imports fail after this, use Run > Restart & clear outputs, then run again.

## 1. Config

In [ ]:
import re, json, time, random
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID      = "Qwen/Qwen2.5-0.5B-Instruct"
OUT_DIR       = "/kaggle/working/grpo-run"
MAX_STEPS     = 1000    # ~23s/step on a T4 -> ~6.5h of training. Use 20 for a quick smoke test.
EVAL_N        = 1319    # full GSM8K test set for the final before/after numbers
PROBE_N       = 200     # smaller subset evaluated during training (accuracy-over-steps plot)
PROBE_EVERY   = 200     # evaluate the probe subset every N steps
SAMPLE_EVERY  = 100     # save demo outputs every N steps (for the "watch it learn" demo)
EVAL_TOKENS   = 512     # eval generation budget; generous so long base answers aren't cut off
SEED          = 42

random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print(torch.cuda.get_device_name(0))

## 2. Data

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful math solver. Think through the problem step by step, then give the final answer. "
    "Respond exactly in this format:\n<think>\n...your reasoning...\n</think>\n<answer>\n...final number only...\n</answer>"
)

def gold_answer(text):
    return text.split("####")[-1].strip().replace(",", "")

USER_SUFFIX = "\n\nShow your reasoning inside <think></think> tags, then give only the final number inside <answer></answer> tags."

def to_prompt(q):
    # small models follow the user turn much better than the system prompt, so repeat the format there
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q + USER_SUFFIX}]

gsm = load_dataset("openai/gsm8k", "main")
train_ds = gsm["train"].map(lambda x: {"prompt": to_prompt(x["question"]), "answer": gold_answer(x["answer"])})
train_ds = train_ds.remove_columns(["question"]).shuffle(seed=SEED)

test_ds = gsm["test"].shuffle(seed=SEED).select(range(EVAL_N))
test_qs = [x["question"] for x in test_ds]
test_gold = [gold_answer(x["answer"]) for x in test_ds]
probe_qs, probe_gold = test_qs[:PROBE_N], test_gold[:PROBE_N]

# fixed questions used for the checkpoint demo
DEMO_QS, DEMO_GOLD = test_qs[:8], test_gold[:8]
print(train_ds[0]["prompt"][1]["content"][:200], "->", train_ds[0]["answer"])

## 3. Reward functions

In [ ]:
NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")

def extract_answer(text):
    # 1) inside <answer> tags, 2) inside \boxed{}, 3) last number anywhere (lenient fallback)
    for pat in (r"<answer>(.*?)</answer>", r"\\boxed\{(.*?)\}"):
        m = re.search(pat, text, re.DOTALL)
        if m:
            nums = NUM_RE.findall(m.group(1).replace(",", ""))
            if nums:
                return nums[-1]
    nums = NUM_RE.findall(text.replace(",", ""))
    return nums[-1] if nums else None

def same_number(a, b):
    try:
        return abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return False

def _texts(completions):
    # conversational format: each completion is [{"role": "assistant", "content": ...}]
    return [c[0]["content"] if isinstance(c, list) else c for c in completions]

def correctness_reward(completions, answer, **kwargs):
    return [2.0 if same_number(extract_answer(t), a) else 0.0 for t, a in zip(_texts(completions), answer)]

FORMAT_RE = re.compile(r"^\s*<think>.*?</think>\s*<answer>.*?</answer>\s*$", re.DOTALL)
def format_reward(completions, **kwargs):
    # partial credit: 0.125 per tag used exactly once (max 0.5), so the model gets nudged toward the format gradually
    tags = ["<think>", "</think>", "<answer>", "</answer>"]
    return [sum(0.125 for tag in tags if t.count(tag) == 1) for t in _texts(completions)]

# quick sanity check
good = [[{"content": "<think>\n2+2=4\n</think>\n<answer>\n4\n</answer>"}]]
bad  = [[{"content": "the answer is 5"}]]
print(correctness_reward(good + bad, answer=["4", "4"]), format_reward(good + bad))  # expect [2.0, 0.0] [0.5, 0.0]

## 4. Eval helpers

In [ ]:
@torch.no_grad()
def generate(model, tok, questions, max_new_tokens=EVAL_TOKENS, bs=32):
    tok.padding_side = "left"
    was_training = model.training
    model.eval()
    outs = []
    for i in range(0, len(questions), bs):
        chunk = questions[i:i+bs]
        texts = [tok.apply_chat_template(to_prompt(q), tokenize=False, add_generation_prompt=True) for q in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id, use_cache=True)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if was_training:
        model.train()
    return outs

def evaluate(model, tok, label, qs=None, gold=None, verbose=True):
    qs, gold = (qs or test_qs), (gold or test_gold)
    t0 = time.time()
    outs = generate(model, tok, qs)
    correct = [same_number(extract_answer(o), g) for o, g in zip(outs, gold)]
    fmt = [bool(FORMAT_RE.match(o)) for o in outs]
    res = {"label": label, "accuracy": sum(correct) / len(correct), "format_rate": sum(fmt) / len(fmt),
           "avg_len_chars": sum(map(len, outs)) / len(outs)}
    print(f"[{label}] n={len(qs)}  acc={res['accuracy']:.1%}  format={res['format_rate']:.1%}  ({time.time()-t0:.0f}s)")
    if verbose:
        print("--- sample output ---")
        print(outs[0][:600])
    return res, outs

## 5. Baseline (before training)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to("cuda")

base_res, base_outs = evaluate(model, tok, "base")   # full test set, ~15-20 min
probe_log = [{"step": 0, "accuracy": sum(same_number(extract_answer(o), g) for o, g in zip(base_outs[:PROBE_N], probe_gold)) / PROBE_N}]
demo_log = [{"step": 0, "outputs": generate(model, tok, DEMO_QS)}]
json.dump({"questions": DEMO_QS, "gold": DEMO_GOLD, "checkpoints": demo_log}, open(f"{OUT_DIR}/demo_samples.json", "w"), indent=2)
json.dump(base_res, open(f"{OUT_DIR}/eval_base.json", "w"), indent=2)

## 6. Callback: demo outputs + accuracy probe during training

In [ ]:
class DemoSampler(TrainerCallback):
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % SAMPLE_EVERY == 0:
            demo_log.append({"step": state.global_step, "outputs": generate(model, tok, DEMO_QS)})
            json.dump({"questions": DEMO_QS, "gold": DEMO_GOLD, "checkpoints": demo_log},
                      open(f"{OUT_DIR}/demo_samples.json", "w"), indent=2)
        if state.global_step % PROBE_EVERY == 0:
            r, _ = evaluate(model, tok, f"probe@{state.global_step}", probe_qs, probe_gold, verbose=False)
            probe_log.append({"step": state.global_step, "accuracy": r["accuracy"], "format_rate": r["format_rate"]})
            json.dump(probe_log, open(f"{OUT_DIR}/probe_log.json", "w"), indent=2)

## 7. GRPO training

In [ ]:
peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",   # v2: higher rank = more capacity
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

args = GRPOConfig(
    output_dir=OUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=5e-5,               # v2: 2.5x v1, LoRA tolerates higher LRs
    lr_scheduler_type="cosine",
    warmup_steps=30,
    per_device_train_batch_size=16,   # counts completions, not prompts
    gradient_accumulation_steps=2,
    num_generations=8,                # group size G: 8 answers per question
    max_completion_length=256,
    temperature=0.9,
    beta=0.04,                        # KL penalty (good ablation knob)
    fp16=True, bf16=False,            # T4 has no bf16
    logging_steps=5,
    save_steps=200,
    save_total_limit=5,
    report_to="none",
    seed=SEED,
)

tok.padding_side = "left"
trainer = GRPOTrainer(
    model=model,
    processing_class=tok,
    reward_funcs=[correctness_reward, format_reward],
    args=args,
    train_dataset=train_ds,
    peft_config=peft_config,
    callbacks=[DemoSampler()],
)
trainer.train()
trainer.save_model(f"{OUT_DIR}/final_adapter")

## 8. Final eval

In [ ]:
final_res, final_outs = evaluate(trainer.model, tok, "grpo")
json.dump({"base": base_res, "grpo": final_res}, open(f"{OUT_DIR}/results.json", "w"), indent=2)
pd.DataFrame({"question": test_qs, "gold": test_gold, "base": base_outs, "grpo": final_outs}).to_csv(
    f"{OUT_DIR}/eval_outputs.csv", index=False)
pd.DataFrame([base_res, final_res])

## 9. Plots

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
hist.to_csv(f"{OUT_DIR}/train_log.csv", index=False)
logs = hist.dropna(subset=["reward"]) if "reward" in hist else hist

def col(name_options):
    for n in name_options:
        if n in logs.columns: return n

panels = [
    ("Total reward", col(["reward"])),
    ("Correctness reward", col(["rewards/correctness_reward/mean", "rewards/correctness_reward"])),
    ("Format reward", col(["rewards/format_reward/mean", "rewards/format_reward"])),
    ("Completion length", col(["completions/mean_length", "completion_length"])),
    ("KL", col(["kl"])),
]
panels = [p for p in panels if p[1]]
fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 3.5))
for ax, (title, c) in zip(axes, panels):
    ax.plot(logs["step"], logs[c].rolling(3, min_periods=1).mean())
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150); plt.show()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
bars = a1.bar(["Base", "GRPO"], [base_res["accuracy"] * 100, final_res["accuracy"] * 100], color=["#999", "#4c72b0"])
a1.bar_label(bars, fmt="%.1f%%"); a1.set_ylabel("accuracy (%)"); a1.set_title(f"GSM8K test ({EVAL_N} questions)")
pl = pd.DataFrame(probe_log)
a2.plot(pl["step"], pl["accuracy"] * 100, marker="o", color="#4c72b0")
a2.set_xlabel("training step"); a2.set_ylabel("accuracy (%)"); a2.set_title(f"Accuracy during training ({PROBE_N}-question probe)")
a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/accuracy.png", dpi=150, bbox_inches="tight"); plt.show()

## 10. Package outputs
Download `grpo_results.zip` from the Output tab. It has the results, plots, demo samples and the LoRA adapter.

In [ ]:
import shutil
keep = "/kaggle/working/grpo_results"
os.makedirs(keep, exist_ok=True)
for f in ["results.json", "eval_outputs.csv", "train_log.csv", "demo_samples.json", "probe_log.json", "training_curves.png", "accuracy.png"]:
    if os.path.exists(f"{OUT_DIR}/{f}"): shutil.copy(f"{OUT_DIR}/{f}", keep)
shutil.copytree(f"{OUT_DIR}/final_adapter", f"{keep}/final_adapter", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/grpo_results", "zip", keep)
print(os.listdir(keep))